# Least squares from scratch

A PROTOTYPE of the compute gate (Prompt 268, COMPUTE_GATE_II.md): it is not in the catalogue and not shipped. It reads its input as the text `gravitas_input` and its settings as the JSON text `gravitas_params`, and returns `gravitas_output`, a dict of numbers. Python only, numpy and scipy only.


In [ ]:
import json
import numpy as np
from scipy.stats import f as fdist
doc = json.loads(gravitas_input)
assert doc.get("format") == "gravitas.observation" and doc.get("kind") == "time-series", "this notebook reads a gravitas.observation/1 time series"
col = {c["id"]: np.array(c["values"], float) for c in doc["columns"]}
n = min(600, len(col["time"]))
assert n >= 5, "refused: the window holds %d points and a degree-4 polynomial needs 5" % n
t, y, e = col["time"][:n], col["flux"][:n], col["flux-error"][:n]
assert bool(np.all(e > 0)), "refused: an uncertainty is not positive"
half = (t.max() - t.min()) / 2
assert half > 0, "refused: every time in the window is the same, the design is rank deficient"
x = (t - (t.max() + t.min()) / 2) / half
w = 1.0 / e ** 2

In [ ]:
sw = np.sqrt(w)
out, chis = {}, []
for d in range(5):
    A = np.vander(x, d + 1, increasing=True) * sw[:, None]
    Q, R = np.linalg.qr(A)
    beta = np.linalg.solve(R, Q.T @ (y * sw))
    Rinv = np.linalg.inv(R)
    cov = Rinv @ Rinv.T
    r = (y - np.vander(x, d + 1, increasing=True) @ beta) * sw
    chi2 = float(r @ r)
    chis.append(chi2)
    out[str(d)] = {"coef": beta.tolist(), "covdiag": np.diag(cov).tolist(), "chi2": chi2, "dof": n - (d + 1)}
tests = []
for d in range(4):
    dof = n - (d + 2)
    F = (chis[d] - chis[d + 1]) / (chis[d + 1] / dof)
    tests.append({"from": d, "to": d + 1, "F": float(F), "p": float(fdist.sf(F, 1, dof)), "rssReduced": chis[d], "rssFull": chis[d + 1], "n": n})
gravitas_output = {"answer": {"n": n, "degrees": out, "F": tests}}